<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 5 &middot; Part 4 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Practice Lab: Neural Networks for Multiclass Classification

Week 4's practice lab told 0s from 1s. This one reads **all ten digits**. The network is the
lecture's, with two changes this week has motivated: ReLU in the hidden layers, and a ten-unit
output layer that produces logits for a softmax.

$$64 \;\to\; 25\ \text{ReLU} \;\to\; 15\ \text{ReLU} \;\to\; 10\ \text{linear (logits)}, \qquad
  \texttt{SparseCategoricalCrossentropy(from\_logits=True)}, \qquad \text{Adam}.$$

You will train it, measure where it fails with a confusion matrix, and then reproduce its
probabilities in NumPy from the trained weights &mdash; ReLU, ReLU, logits, stable softmax &mdash;
so that nothing between the pixels and the answer is hidden.

This notebook is the **assessed practice lab** for Week 5. The two checkpoints and the exercises
at the end are what you hand in.

**Companion theory lecture:** CSE 815, Week 5, Part 1 (Training Neural Networks) and Part 2
(Multiclass Classification and Back-propagation).

## What you will be able to do

1. Prepare a ten-class image data set and split it into training and test sets.
2. Build and train a multiclass network with a linear output layer and `from_logits=True`.
3. Turn logits into class predictions and into probabilities.
4. Build and read a confusion matrix, and find the classes the model confuses.
5. Reproduce the trained network's probabilities in NumPy with a numerically stable softmax.

---

## 1. Setup and the data

`load_digits()` holds 1797 images of the digits 0&ndash;9, each $8 \times 8$ pixels with
brightness 0&ndash;16. As in Week 4, divide by 16 to put every feature in $[0, 1]$. The labels are
integers 0&ndash;9, exactly the form `SparseCategoricalCrossentropy` expects.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

from sklearn.datasets import load_digits

digits = load_digits()
images = digits.images                    # (1797, 8, 8)
X = digits.data / 16.0                    # (1797, 64)
y = digits.target                         # (1797,) integers 0..9

print("X:", X.shape, "  y:", y.shape, "  dtype of y:", y.dtype)
print("images per digit:", np.bincount(y))
assert X.shape == (1797, 64) and set(np.unique(y)) == set(range(10))

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(10, 3.3))
for d in range(10):
    for r, i in enumerate(np.flatnonzero(y == d)[:3]):
        axes[r, d].imshow(images[i], cmap="gray_r")
        axes[r, d].axis("off")
    axes[0, d].set_title(str(d), fontsize=10)
fig.suptitle("Three examples of each digit", fontsize=10)
plt.tight_layout()
plt.show()

---

## 2. Training and test sets

Hold back 360 images, a fifth of the data, for testing. With ten classes it is worth checking
that every digit appears in both sets in roughly the same proportion.

In [ ]:
rng = np.random.default_rng(816)
idx = rng.permutation(len(y))
train, test = idx[:1437], idx[1437:]
X_train, y_train, X_test, y_test = X[train], y[train], X[test], y[test]

print("X_train", X_train.shape, "  X_test", X_test.shape)
print("train share per digit:", np.round(np.bincount(y_train) / len(y_train), 3))
print("test share per digit :", np.round(np.bincount(y_test) / len(y_test), 3))
assert len(set(train) & set(test)) == 0 and len(y_test) == 360
assert np.bincount(y_test, minlength=10).min() >= 25

---

## 3. Build the network

The output layer has one unit per class and a **linear** activation: it outputs the logits
$z_0, \dots, z_9$, and the loss applies the softmax itself, stably (Part 2).

In [ ]:
keras.utils.set_random_seed(816)
model = Sequential([
    Input(shape=(64,)),
    Dense(units=25, activation="relu", name="L1"),
    Dense(units=15, activation="relu", name="L2"),
    Dense(units=10, activation="linear", name="L3"),
], name="digits_0_to_9")
model.summary()

assert [layer.count_params() for layer in model.layers] == [64 * 25 + 25, 25 * 15 + 15, 15 * 10 + 10]
assert model.count_params() == 1625 + 390 + 160 == 2175
print("1625 + 390 + 160 = 2175 parameters; only the last layer changed from Week 4 (16 -> 160).")

---

## 4. Train

In [ ]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
)
history = model.fit(X_train, y_train, epochs=100, verbose=0)

loss = history.history["loss"]
for epoch in [1, 5, 20, 50, 100]:
    print(f"epoch {epoch:3d}   cost {loss[epoch - 1]:.4f}")
print(f"\ncost of guessing uniformly over 10 classes: -log(1/10) = {np.log(10):.4f}")

fig, ax = plt.subplots(figsize=(6.0, 3.4))
ax.plot(np.arange(1, 101), loss, c=BLUE, lw=2)
ax.axhline(np.log(10), c=GRAY, ls="--", lw=1, label="uniform guess, log 10")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("training cost")
ax.legend(fontsize=8); ax.set_title("Training cost", fontsize=10)
plt.show()

An untrained ten-class softmax that knows nothing assigns about $1/10$ to every class, for a cost
of $\log 10 \approx 2.30$. The first epoch's cost should be near that; if it is far above, the
initial logits are badly scaled.

---

## 5. Predictions: logits, classes and probabilities

In [ ]:
logits_test = model.predict(X_test, verbose=0)            # (360, 10)
probs_test = tf.nn.softmax(logits_test).numpy()
yhat_test = logits_test.argmax(axis=1)                    # softmax does not change the argmax

assert np.array_equal(yhat_test, probs_test.argmax(axis=1))
assert np.allclose(probs_test.sum(axis=1), 1, atol=1e-5)

acc_train = float((model.predict(X_train, verbose=0).argmax(axis=1) == y_train).mean())
acc_test = float((yhat_test == y_test).mean())
print(f"training accuracy: {acc_train:.4f}")
print(f"test accuracy    : {acc_test:.4f}   ({int((yhat_test != y_test).sum())} of 360 wrong)")
assert acc_test >= 0.9, "unexpectedly poor -- re-run the build and train cells"

### One image, ten probabilities

In [ ]:
k = 0
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), gridspec_kw={"width_ratios": [1, 3]})
axes[0].imshow(X_test[k].reshape(8, 8), cmap="gray_r"); axes[0].axis("off")
axes[0].set_title(f"label {y_test[k]}", fontsize=10)
axes[1].bar(range(10), probs_test[k], color=[ROSE if d == yhat_test[k] else BLUE for d in range(10)])
axes[1].set_xticks(range(10)); axes[1].set_ylim(0, 1)
axes[1].set_xlabel("digit"); axes[1].set_ylabel("softmax probability")
plt.tight_layout()
plt.show()
print("logits       :", logits_test[k].round(2))
print("probabilities:", probs_test[k].round(4))

---

## 6. Where does it go wrong? The confusion matrix

Accuracy is one number for ten classes. A **confusion matrix** $C$ counts, for every pair, how
often an image of digit $i$ was predicted as digit $j$: $C_{ij}$. The diagonal holds the correct
predictions; everything off it is a specific mistake.

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes=10):
    C = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
    return C


C = confusion_matrix(y_test, yhat_test)
print(C)
assert C.sum() == 360 and np.trace(C) == int((yhat_test == y_test).sum())

from sklearn.metrics import confusion_matrix as sk_confusion
assert np.array_equal(C, sk_confusion(y_test, yhat_test))

fig, ax = plt.subplots(figsize=(4.8, 4.2))
im = ax.imshow(C, cmap="Blues")
for i in range(10):
    for j in range(10):
        if C[i, j]:
            ax.text(j, i, C[i, j], ha="center", va="center", fontsize=8,
                    color="white" if C[i, j] > C.max() / 2 else "black")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("predicted digit"); ax.set_ylabel("true digit")
ax.set_title("Test-set confusion matrix", fontsize=10)
plt.show()

In [ ]:
off = C.copy()
np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], i, j) for i in range(10) for j in range(10) if off[i, j]), reverse=True)
print("most frequent mistakes (count: true -> predicted):")
for n, i, j in pairs[:5]:
    print(f"  {n}: {i} -> {j}")

wrong = np.flatnonzero(yhat_test != y_test)
fig, axes = plt.subplots(1, min(10, len(wrong)), figsize=(10, 1.9), squeeze=False)
for ax, i in zip(axes[0], wrong[:10]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r"); ax.axis("off")
    ax.set_title(f"{y_test[i]} as {yhat_test[i]}\nP={probs_test[i, yhat_test[i]]:.2f}", fontsize=8)
fig.suptitle("Misclassified test images: true as predicted", fontsize=10)
plt.tight_layout()
plt.show()

In the reference run 9 of the 360 test images are wrong, and the only mistake made twice is a
5 read as a 9. Look at the misclassified images before blaming the model. At $8 \times 8$ pixels some of them
are genuinely ambiguous; others are clear to you and still wrong &mdash; those are the
interesting ones.

---

## 7. The trained network in NumPy

Pull out the six trained arrays and run the network yourself. Three changes from Week 4's
`my_dense`: the hidden layers use ReLU, the output layer is linear, and the probabilities come
from a softmax with the maximum subtracted.

In [ ]:
def relu(z):
    return np.maximum(0.0, z)


def stable_softmax(Z):
    Zs = Z - Z.max(axis=1, keepdims=True)
    EZ = np.exp(Zs)
    return EZ / EZ.sum(axis=1, keepdims=True)


W1, b1, W2, b2, W3, b3 = model.get_weights()
print("shapes:", [w.shape for w in model.get_weights()])

A1 = relu(X_test @ W1 + b1)          # (360, 25)
A2 = relu(A1 @ W2 + b2)              # (360, 15)
Z3 = A2 @ W3 + b3                    # (360, 10) logits
P_np = stable_softmax(Z3)

print(f"largest |logit difference|      : {np.abs(Z3 - logits_test).max():.1e}")
print(f"largest |probability difference|: {np.abs(P_np - probs_test).max():.1e}")
assert np.allclose(Z3, logits_test, atol=1e-4)
assert np.allclose(P_np, probs_test, atol=1e-6)
assert np.array_equal(Z3.argmax(axis=1), yhat_test)
print("NumPy and TensorFlow: the same 360 predictions.")

How many hidden units are actually used? A ReLU unit that is zero on every training image is
dead (Part 1).

In [ ]:
H1 = relu(X_train @ W1 + b1)
H2 = relu(H1 @ W2 + b2)
print(f"layer 1: {int((H1 > 0).any(axis=0).sum())} of 25 units active on some training image")
print(f"layer 2: {int((H2 > 0).any(axis=0).sum())} of 15 units active on some training image")
print(f"share of zero activations -- layer 1: {(H1 == 0).mean():.2f}, layer 2: {(H2 == 0).mean():.2f}")

### Checkpoint 1

Write `my_predict_proba(X, params)` that takes a batch `X` of shape `(m, 64)` and
`params = model.get_weights()`, and returns softmax probabilities of shape `(m, 10)` &mdash;
any number of `Dense` layers, ReLU on all but the last. Then:

1. Assert that it matches `tf.nn.softmax(model.predict(X_test))` to `1e-6`.
2. Compute the **per-digit test accuracy** $C_{ii} / \sum_j C_{ij}$ from the confusion matrix
   and report the best- and worst-recognised digits.
3. Find the test image whose largest probability is smallest &mdash; the model's least
   confident prediction &mdash; and display it with its ten probabilities.

*Expected (reference machine):* identical probabilities; per-digit accuracy from `0.889`
(digit 5, the hardest) to `1.000` (six digits are perfect); the least confident prediction is a
9 whose largest probability is only about `0.41`, spread between 9, 7 and 8.

In [ ]:
# Your code here

### Checkpoint 2

How much did this week's changes matter? Rebuild the network twice more, each time after
`keras.utils.set_random_seed(816)`, train exactly as in section 4, and report the test accuracy
and the training cost at epochs 5 and 100:

1. **Week 4 style:** `"sigmoid"` in both hidden layers (output and loss unchanged).
2. **Fragile output:** ReLU hidden layers, but `activation="softmax"` in the output layer and
   `SparseCategoricalCrossentropy()` without `from_logits`.

*Expected (reference machine):* the sigmoid network learns much more slowly &mdash; cost
`2.20` at epoch 5 (barely below $\log 10$) against `0.93` with ReLU, and `0.106` against
`0.009` at epoch 100 &mdash; and reaches test accuracy `0.964` against `0.975`. The fragile
version matches the recommended one to four decimals here (`0.9324`, `0.0088`, `0.975`).
Explain both results using Parts 1 and 2: why does the clipping of Part 2 never bite on this
data?

In [ ]:
# Your code here

---

## 8. Recap

- Ten classes need ten output units. Make them **linear** and use
  `SparseCategoricalCrossentropy(from_logits=True)`; labels stay as integers 0&ndash;9.
- An uninformed ten-class model has cost $\log 10 \approx 2.30$; training drives it far below.
- `argmax` of the logits gives the class; `tf.nn.softmax` gives probabilities when you need them.
- The $64 \to 25 \to 15 \to 10$ network has `2175` parameters. In the reference run it fits
  the training set perfectly and classifies `0.975` of the test images correctly (9 of 360
  wrong); 5 is the digit it finds hardest.
- A confusion matrix shows *which* mistakes are made; the per-digit accuracy is its diagonal
  divided by its row sums.
- ReLU, ReLU, a linear layer and a stable softmax in NumPy reproduce the trained model's
  probabilities exactly. There is no hidden step.

### Exercises to hand in

Hand in this notebook with both checkpoints completed and the following, each answered in a
markdown cell beneath your code.

1. **Fix the worst pair.** Take the most frequent mistake in your confusion matrix. Show the
   class-mean images of the two digits and their difference, and propose one change &mdash;
   architecture, data, or input features &mdash; that might reduce that mistake. Try it and
   report the new confusion count.
2. **Width and depth.** Train networks with hidden layers $(10)$, $(25)$, $(25, 15)$,
   $(64, 32)$ and $(64, 32, 16)$. Tabulate parameters, training accuracy and test accuracy.
   Which is the smallest network within one percentage point of the best test accuracy?
3. **Learning rate.** Train the section-3 network with Adam at learning rates $10^{-4}$,
   $10^{-3}$, $10^{-2}$ and $10^{-1}$. Plot the four learning curves on one axis and explain
   each shape.
4. **Calibration.** Group the test predictions by their largest softmax probability into bins
   $[0.5, 0.7), [0.7, 0.9), [0.9, 0.99), [0.99, 1]$. For each bin report how many predictions
   fall in it and what fraction of them are correct. Is a probability of 0.9 from this model
   right 90% of the time?
5. **Your own back-prop.** Using Part 3's method, write `backward` for this architecture
   (two ReLU layers, softmax output; $\partial L/\partial z_j = a_j - [j = y]$), pass a
   gradient check on 20 training images, and train the network for 2000 steps of Adam with your
   own code. Compare its test accuracy with TensorFlow's.

### Next

**Week 6 &mdash; Advice for applying machine learning:** training, validation and test sets,
diagnosing bias and variance, and deciding what to try next when a model is not good enough.